# AI-Assisted Insurance Claims Processing: Walk-through

This notebook walks through the claims pipeline step by step (Tasks 1–7). All logic lives in the `claims` package, and the notebook only calls it and shows the results.

**The LLM extracts, classifies, gathers evidence and writes emails. Python decides and calculates.**

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from claims.config import load_settings
from claims.llm.client import GroqLLMClient
from claims.ingestion import load_dataset

settings = load_settings()
llm = GroqLLMClient(settings)
emails = {e.email_id: e for e in load_dataset(settings.dataset_path)}
print(len(emails), 'emails loaded. Model:', settings.model)
print('Ground-truth label kept aside (never sent to the LLM):', emails['EMAIL-01'].source_label)

10 emails loaded. Model: openai/gpt-oss-120b
Ground-truth label kept aside (never sent to the LLM): Home Insurance | Complete


## Task 1: Understand the customer email (Pydantic structured output)
The extraction prompt is RTCO with an injection guard. Python then checks that the name, policy number and amount appear verbatim in the email, and parses the amount as a `Decimal`.

In [2]:
from claims.extraction import ClaimExtractor

email = emails['EMAIL-01']
print(email.full_text)
extraction = ClaimExtractor(llm).extract(email)
print(extraction.data.model_dump_json(indent=2))
print('Parsed amount:', extraction.claimed_amount, '| Parsed date:', extraction.event_date, '| Grounding issues:', extraction.grounding_issues)

Subject: Water damage in my bathroom

Hello,

My policy number is HOME-2026-1001.

On 25 October 2026, my geyser burst in the bathroom at my residence in Mumbai, causing water damage.

The bathroom tiles and the wooden cabinet were damaged by the water.

The estimated repair cost is Rs. 45,000. I have attached photographs of the damage and the plumber's estimate.

Please help me file a claim.

Regards,
Neha Kapoor


{
  "customer_name": "Neha Kapoor",
  "policy_number": "HOME-2026-1001",
  "event_date": "2026-10-25",
  "event_location": "Mumbai",
  "event_description": "My geyser burst in the bathroom, causing water damage.",
  "loss_damage_details": "The bathroom tiles and the wooden cabinet were damaged by the water.",
  "claimed_amount_text": "Rs. 45,000",
  "supporting_documents": [
    "photographs of the damage",
    "plumber's estimate"
  ],
  "pending_documents": [],
  "customer_request": "Please help me file a claim.",
  "intent": "new_claim",
  "vehicle_registration": null,
  "fir_number": null,
  "hospital_name": null,
  "admission_date": null,
  "discharge_date": null,
  "insured_person_name": null,
  "claimant_relationship": null,
  "ambiguities": []
}
Parsed amount: 45000 | Parsed date: 2026-10-25 | Grounding issues: []


## Task 2: Identify missing information (Python rules CMP-001…006)

In [3]:
from claims.completeness import check_completeness

def show_completeness(ext):
    c = check_completeness(ext, settings.processing_date)
    for f in c.fields:
        print(f'{f.label:<24} {f.status.value}')
    print('Complete:', c.is_complete)
    return c

completeness = show_completeness(extraction)

Customer name            AVAILABLE
Policy number            AVAILABLE
Event date               AVAILABLE
Event description        AVAILABLE
Loss / damage details    AVAILABLE
Claim amount             AVAILABLE
Complete: True


## Task 3: Follow-up email when mandatory details are missing (EMAIL-03 has no policy number)

In [4]:
from claims.workflow import ClaimsProcessor

processor = ClaimsProcessor(llm, settings)
pending = processor.process(emails['EMAIL-03'])
show_completeness(pending.extraction)
print('\nOutcome:', pending.decision.outcome.value)
print('\n' + pending.communication.subject + '\n\n' + pending.communication.body)

Customer name            AVAILABLE
Policy number            MISSING
Event date               AVAILABLE
Event description        AVAILABLE
Loss / damage details    AVAILABLE
Claim amount             AVAILABLE
Complete: False

Outcome: PENDING

Claim CLM-A2B00BB0 – Additional Information Required

Dear Deepa Iyer,

We are processing your claim (reference: CLM-A2B00BB0) regarding the electrical short circuit fire in your living room. To continue, we need you to confirm the following details:

1. Customer name
2. Policy number (missing)
3. Event date
4. Event description
5. Loss / damage details
6. Claim amount

Please reply to this email with the missing information, quoting the claim reference above.

Thank you for your cooperation.

Claims Team


## Task 4: Classify the claim (zero-shot vs few-shot)
The few-shot prompt uses synthetic examples, not dataset emails. The category is accepted only when both methods agree with confidence ≥ 0.7. Otherwise the claim is flagged for review rather than forced into a category.

In [5]:
from claims.classification import ClaimClassifier
from claims.llm.prompts import FEW_SHOT_EXAMPLES

classifier = ClaimClassifier(llm, settings.classification_confidence_threshold)
print('Few-shot examples:', len(FEW_SHOT_EXAMPLES))
for eid in ['EMAIL-01', 'EMAIL-04', 'EMAIL-07', 'EMAIL-09']:
    ext = ClaimExtractor(llm).extract(emails[eid])
    out = classifier.classify(ext)
    print(f"{eid}: zero-shot={out.zero_shot.insurance_type} ({out.zero_shot.confidence:.2f}) | "
          f"few-shot={out.few_shot.insurance_type} ({out.few_shot.confidence:.2f}) | final={out.final_type}")

Few-shot examples: 6


EMAIL-01: zero-shot=Home Insurance (0.99) | few-shot=Home Insurance (0.99) | final=Home Insurance


EMAIL-04: zero-shot=Vehicle Insurance (0.95) | few-shot=Vehicle Insurance (0.95) | final=Vehicle Insurance


EMAIL-07: zero-shot=Medical Insurance (0.99) | few-shot=Medical Insurance (0.99) | final=Medical Insurance


EMAIL-09: zero-shot=Life Insurance (0.99) | few-shot=Life Insurance (0.99) | final=Life Insurance


The full 10-email comparison is written to `outputs/classification_report.md` by `python -m claims run`.

In [6]:
report = ROOT / 'outputs' / 'classification_report.md'
print(report.read_text().split('## Reasons')[0] if report.exists() else 'Run `python -m claims run` first.')

# Classification: zero-shot vs few-shot

- Zero-shot accuracy vs dataset labels: **10/10**
- Few-shot accuracy vs dataset labels: **10/10**
- Methods agreed: **10/10**
- Reliable (agree and confidence above threshold): **10/10**

| Email | Expected | Zero-shot (conf.) | Few-shot (conf.) | Agree | Final |
|---|---|---|---|---|---|
| EMAIL-01 | Home Insurance | Home Insurance (0.99) | Home Insurance (0.99) | yes | Home Insurance |
| EMAIL-02 | Home Insurance | Home Insurance (0.99) | Home Insurance (0.98) | yes | Home Insurance |
| EMAIL-03 | Home Insurance | Home Insurance (0.99) | Home Insurance (0.99) | yes | Home Insurance |
| EMAIL-04 | Vehicle Insurance | Vehicle Insurance (0.95) | Vehicle Insurance (0.90) | yes | Vehicle Insurance |
| EMAIL-05 | Vehicle Insurance | Vehicle Insurance (0.99) | Vehicle Insurance (0.99) | yes | Vehicle Insurance |
| EMAIL-06 | Vehicle Insurance | Vehicle Insurance (0.99) | Vehicle Insurance (0.99) | yes | Vehicle Insurance |
| EMAIL-07 | Medical Insur

## Task 5: Verify against the policy dictionaries
The policy dictionary is stored verbatim in `claims/policy/catalog.py`. The LLM may only pick coverage and exclusion names from the policy's own lists, and must quote the email. The deterministic validator verifies the quotes and applies the rules.

In [7]:
from claims.policy.catalog import INSURANCE_POLICIES
print(json.dumps(INSURANCE_POLICIES['Vehicle Insurance'], indent=2))

{
  "covered_claim_categories": [
    "Accidental vehicle damage",
    "Vehicle theft",
    "Fire damage",
    "Natural disaster damage",
    "Windshield and glass damage"
  ],
  "coverage_limits": {
    "vehicle": 800000
  },
  "deductible": 2000,
  "exclusions": [
    "Driving without a valid licence",
    "Driving under the influence",
    "Normal wear and tear",
    "Mechanical breakdown"
  ],
  "claim_requirements": [
    "Policy number",
    "Vehicle registration number",
    "Incident date",
    "Photos of damage",
    "Repair estimate",
    "FIR for theft claims"
  ]
}


In [8]:
rejected = processor.process(emails['EMAIL-04'])
print('Evidence:', rejected.policy_match.model_dump_json(indent=2))
for r in rejected.validations:
    print(f'{r.rule_id:<8} {r.status.value:<15} {r.explanation}')
print('\nOutcome:', rejected.decision.outcome.value, '-', rejected.decision.rationale)

Evidence: {
  "matched_coverages": [],
  "coverage_evidence": null,
  "exclusion_findings": [
    {
      "exclusion": "Mechanical breakdown",
      "evidence": "engine had suffered a mechanical breakdown due to wear and tear"
    }
  ],
  "reason": "The email describes a mechanical breakdown of the engine, which is listed as an exclusion, so no covered categories apply."
}
POL-001  PASSED          Policy number matches the format <HOME|VEH|MED|LIFE>-YYYY-NNNN.
POL-002  PASSED          Policy number belongs to Vehicle Insurance, consistent with the claim.
POL-003  NOT_APPLICABLE  No policy register (holders, status, dates) was supplied; this check cannot be performed.
CLS-001  PASSED          Zero-shot and few-shot agree on Vehicle Insurance with confidence >= 0.7.
COV-001  NEEDS_REVIEW    The incident could not be matched to any covered category of the policy.
EXC-001  FAILED          The customer's own description matches the policy exclusion(s) "Mechanical breakdown".
DOC-001  PASSE

## Task 6: Calculate the claim amount in Python
`Potential Payout = max(0, min(Eligible Loss, Coverage Limit) − Deductible)` with `Decimal` arithmetic.

In [9]:
from decimal import Decimal
from claims.payout import calculate_payout

demo = calculate_payout(claimed_amount=Decimal('85000'), coverage_limit_name='household_items',
                        coverage_limit=Decimal('500000'), deductible=Decimal('5000'))
print('\n'.join(demo.steps))   # worked example -> Rs. 80,000

Eligible loss = claimed amount = Rs. 85,000.00
Coverage limit (household_items) = Rs. 5,00,000.00
min(eligible loss, coverage limit) = Rs. 85,000.00
Deductible = Rs. 5,000.00
Rs. 85,000.00 - Rs. 5,000.00 = Rs. 80,000.00
Potential payout = Rs. 80,000.00


## Task 7: Customer updates (approval and rejection)
The LLM writes the prose from verified facts only. Each draft is checked for the claim reference, the amounts (all must be computed values), the outcome wording and the requested items. If any check fails, a deterministic template is used instead.

In [10]:
approved = processor.process(emails['EMAIL-01'])
print('\n'.join(approved.payout.steps))
print('\n--- generated by', approved.communication.generated_by, '---')
print(approved.communication.subject + '\n\n' + approved.communication.body)

Eligible loss = claimed amount = Rs. 45,000.00
Coverage limit (household_items) = Rs. 5,00,000.00
min(eligible loss, coverage limit) = Rs. 45,000.00
Deductible = Rs. 5,000.00
Rs. 45,000.00 - Rs. 5,000.00 = Rs. 40,000.00
Potential payout = Rs. 40,000.00

--- generated by llm ---
Your Home Insurance Claim – Approved (Reference CLM-8B285C76)

Dear Neha Kapoor,

We are pleased to inform you that your claim (Reference: CLM-8B285C76) for the water damage caused by a burst geyser has been approved.

**Claim Details**
- Policy Number: HOME-2026-1001
- Insurance Type: Home Insurance
- Incident Summary: My geyser burst in the bathroom at my residence in Mumbai, causing water damage.
- Claimed Amount: Rs. 45,000.00
- Coverage Limit: Rs. 5,00,000.00 (household items)
- Deductible: Rs. 5,000.00
- Matched Coverage: Accidental water leakage from burst pipes

**Payout**
- Approved Potential Payout: Rs. 40,000.00

Our claims team will contact you shortly to complete the settlement process.

Thank you f

In [11]:
print('--- generated by', rejected.communication.generated_by, '---')
print(rejected.communication.subject + '\n\n' + rejected.communication.body)

--- generated by llm ---
Claim CLM-3C06FC8D – Decision

Dear Karan Mehta,

We have completed the review of your claim (reference: CLM-3C06FC8D) for your vehicle insurance policy VE​H-2026-2001.

After careful assessment, the claim has been REJECTED. The reason is that the incident you described – the engine stopping while driving – falls under the policy exclusion "Mechanical breakdown". This exclusion applies because the engine failure was due to wear and tear, as stated in your own description.

If you believe any of the information we relied on is incorrect, please reply to this email with the correct details and any supporting documents.

Thank you for your understanding.

Claims Team


## Results for all 10 dataset emails
Produced by `python -m claims run` (see `outputs/summary.md`).

In [12]:
summary = ROOT / 'outputs' / 'summary.md'
print(summary.read_text() if summary.exists() else 'Run `python -m claims run` first.')

# Claims processing summary

| Email | Dataset label | Type | Outcome | Internal | Payout | Key reason | Email draft |
|---|---|---|---|---|---|---|---|
| EMAIL-01 | Home Insurance / Complete | Home Insurance | **APPROVED** | AUTO_DECIDED | Rs. 40,000.00 | all rules passed | APPROVAL (llm) |
| EMAIL-02 | Home Insurance / Complete | Home Insurance | **PENDING** | AUTO_DECIDED | - | DOC-004 | PENDING_OUTSTANDING_ITEMS (llm) |
| EMAIL-03 | Home Insurance / Missing policy number | - | **PENDING** | AUTO_DECIDED | - | CMP-002 | PENDING_MISSING_DETAILS (llm) |
| EMAIL-04 | Vehicle Insurance / Complete - Rejected (exclusion) | Vehicle Insurance | **REJECTED** | AUTO_DECIDED | - | EXC-001 | REJECTION (llm) |
| EMAIL-05 | Vehicle Insurance / Complete | Vehicle Insurance | **APPROVED** | AUTO_DECIDED | Rs. 6,48,000.00 | all rules passed | APPROVAL (llm) |
| EMAIL-06 | Vehicle Insurance / Missing claimed amount | - | **PENDING** | AUTO_DECIDED | - | CMP-006 | PENDING_MISSING_DETAILS (llm) |
| EMA